# Extract PHOENIX poses

Use a Kaggle CPU session with Internet enabled and the PHOENIX image dataset attached. Run the sample check before extracting the full dataset. The output is one `.npz` file per sequence, with a `pose` array of shape `[T, 183]`.


## 1. Configure paths


In [ ]:
INPUT_DIR = "/kaggle/input/datasets/thesisacc2/rwth-phoenix-2014-t"
OUTPUT_DIR = "/kaggle/working/poses"
CHECK_DIR = "/kaggle/working/poses_check"
WORKERS = 0  # Automatically choose the worker count.
NUM_SHARDS = 1
SHARD = 0


## 2. Install extraction dependencies

This uses the MediaPipe version used by the project. Restart the kernel if the runtime asks, then continue from the configuration cell.


In [ ]:
%pip uninstall -y -q tensorflow tensorflow-cpu tf-keras keras mediapipe mediapipe-nightly
%pip install -q "mediapipe==0.10.14" opencv-python-headless tqdm


## 3. Load the repository

An existing checkout is reused. To use a different revision, update that checkout explicitly.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/linhxm/Sign-Language-REL.git"
CODE_DIR = Path("/kaggle/working/Sign-Language-REL")
if not CODE_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(CODE_DIR)], check=True)
assert (CODE_DIR / "main.py").is_file(), f"Invalid repository: {CODE_DIR}"
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
os.environ["SLT_CODE_DIR"] = str(CODE_DIR)
print("Repository:", CODE_DIR)


## 4. Extract a small sample


In [ ]:
subprocess.run([
    sys.executable, "data/extract_poses.py",
    "--input_dir", INPUT_DIR, "--out_dir", CHECK_DIR, "--limit", "5",
], check=True)


## 5. Check the sample

Inspect the shape and zero fraction. A nearly all-zero sequence can indicate missing landmark detections.


In [ ]:
import numpy as np

samples = sorted(Path(CHECK_DIR).glob("*.npz"))
assert samples, "No pose files were produced. Check INPUT_DIR."
for path in samples:
    with np.load(path) as data:
        pose = data["pose"]
    assert pose.ndim == 2 and pose.shape[1] == 183, (path, pose.shape)
    print(path.name, pose.shape, pose.dtype, "zero fraction:", float((pose == 0).mean()))


## 6. Extract the full dataset

Existing pose files are skipped. Keep the output files to resume; a new Kaggle session does not automatically restore the previous working directory.


In [ ]:
subprocess.run([
    sys.executable, "data/extract_poses.py",
    "--input_dir", INPUT_DIR, "--out_dir", OUTPUT_DIR,
    "--workers", str(WORKERS), "--num_shards", str(NUM_SHARDS), "--shard", str(SHARD),
], check=True)


## 7. Save poses for training

Save the output as a Kaggle dataset and attach it to [02_train.ipynb](02_train.ipynb). For a later extraction session, restore existing `.npz` files into `OUTPUT_DIR` first. Keep the extraction log, including missing-landmark counts.


In [ ]:
print("Pose files:", len(list(Path(OUTPUT_DIR).glob("*.npz"))))
